In [ ]:
!pip install -q transformers

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed

from google.colab import files
files.upload()

Saving english_training_final.csv to english_training_final (3).csv
Saving mongolian_expanded_FINAL.csv to mongolian_expanded_FINAL (3).csv


{'english_training_final (3).csv': b'\xef\xbb\xbftext,label,source\nWe just announced that the first participants in each age cohort have been dosed in the Phase 2 study of our mRNA vaccine (mRNA-1273) against novel coronavirus. Read more: https://t.co/woPlKz1bZC #mRNA https://t.co/9VGUoJu5cS,0,Constraint\nWe just announced that we have shipped vials of mRNA-1273 the Company\xe2\x80\x99s vaccine against the novel coronavirus to the NIH to be used in the planned Phase 1 study in the U.S. https://t.co/jNarQEkGhr https://t.co/o7EoaYeQH1,0,Constraint\nWe are delighted that 78 high- and upper-middle countries and economies have now confirmed they will participate in the COVAX Facility and the number is growing. I urge those who have not yet joined to do so by the 18th of September-@DrTedros #COVID19,0,Constraint\nCDC Recommends Mothers Stop Breastfeeding To Boost Vaccine Efficacy,1,Constraint\nScientists at AstraZeneca complain their work on a coronavirus vaccine keeps being delayed by Nodd

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
covid_words = ["covid", "coronavirus", "sars-cov-2", "corona virus", "pandemic"]

def clean(t):
    t = re.sub(r"http\S+|www\.\S+", " ", str(t))
    t = re.sub(r"<[^>]+>", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def load(path):
    df = pd.read_csv(path)
    df["text"] = df["text"].apply(clean)
    df = df[df["text"].str.len() > 0].drop_duplicates("text").reset_index(drop=True)
    df["label"] = df["label"].astype(int)
    return df

def has_covid(t):
    t = t.lower()
    return any(w in t for w in covid_words)

english = load("english_training_final.csv")
mongolian = load("mongolian_expanded_FINAL.csv")
print("English:", len(english), " Mongolian:", len(mongolian))

English: 1111  Mongolian: 495


In [ ]:
class Claims(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True, padding="max_length", max_length=128)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(int(self.labels[i]))
        return item

def train(train_df, seed, start_from="xlm-roberta-base"):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(start_from, num_labels=2)
    args = TrainingArguments(output_dir="tmp", num_train_epochs=4,
                             per_device_train_batch_size=16, per_device_eval_batch_size=32,
                             learning_rate=2e-5, logging_steps=50, save_strategy="no",
                             seed=seed, report_to=[])
    trainer = Trainer(model=model, args=args, train_dataset=Claims(train_df["text"], train_df["label"]))
    trainer.train()
    return model

def score(model, test_df):
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    ds = Claims(test_df["text"], test_df["label"])
    loader = torch.utils.data.DataLoader(ds, batch_size=32)
    preds, gold = [], []
    with torch.no_grad():
        for batch in loader:
            y = batch.pop("labels")
            batch = {k: v.to(device) for k, v in batch.items()}
            preds += model(**batch).logits.argmax(-1).cpu().tolist()
            gold += y.tolist()
    macro = f1_score(gold, preds, average="macro")
    p, r, f, _ = precision_recall_fscore_support(gold, preds, labels=[0, 1], zero_division=0)
    print("  macro-F1: %.4f | accuracy: %.4f" % (macro, accuracy_score(gold, preds)))
    print("  misinfo recall: %.4f | reliable recall: %.4f" % (r[1], r[0]))
    print("  confusion:", confusion_matrix(gold, preds, labels=[0, 1]).tolist())
    return macro

In [ ]:
en_train, en_test = train_test_split(english, test_size=0.20, stratify=english["label"], random_state=42)
mn_pool, mn_test = train_test_split(mongolian, test_size=0.40, stratify=mongolian["label"], random_state=42)
print("EN train/test:", len(en_train), len(en_test), " MN pool/test:", len(mn_pool), len(mn_test))

en_model = train(en_train, 42)

print("\nEnglish ceiling:")
ceiling = score(en_model, en_test)

print("\nRQ1 zero-shot Mongolian:")
zero_shot = score(en_model, mn_test)

en_model.save_pretrained("en_model")

EN train/test: 888 223  MN pool/test: 297 198


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,0.673609
100,0.550448
150,0.412820
200,0.333635



English ceiling:
  macro-F1: 0.8647 | accuracy: 0.8655
  misinfo recall: 0.8898 | reliable recall: 0.8381
  confusion: [[88, 17], [13, 105]]

RQ1 zero-shot Mongolian:
  macro-F1: 0.4254 | accuracy: 0.4697
  misinfo recall: 0.7475 | reliable recall: 0.1919
  confusion: [[19, 80], [25, 74]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import gc

seeds = [42, 123, 2024]
sizes = [0, 20, 40, 80, 100]
sweep = {}

for n in sizes:
    runs = []
    for seed in seeds:
        if n == 0:
            print("\nsize 0, seed %d (zero-shot):" % seed)
            runs.append(score(en_model, mn_test))
        else:
            sub, _ = train_test_split(mn_pool, train_size=n, stratify=mn_pool["label"], random_state=seed)
            m = train(sub, seed, start_from="en_model")
            print("\nsize %d, seed %d:" % (n, seed))
            runs.append(score(m, mn_test))
            del m; gc.collect(); torch.cuda.empty_cache()
    sweep[n] = (np.mean(runs), np.std(runs))

print("\n=== SWEEP (mean over seeds) ===")
print("English ceiling: %.4f" % ceiling)
for n, (mean, std) in sweep.items():
    print("MN %3d -> %.4f ± %.4f" % (n, mean, std))


size 0, seed 42 (zero-shot):
  macro-F1: 0.4254 | accuracy: 0.4697
  misinfo recall: 0.7475 | reliable recall: 0.1919
  confusion: [[19, 80], [25, 74]]

size 0, seed 123 (zero-shot):
  macro-F1: 0.4254 | accuracy: 0.4697
  misinfo recall: 0.7475 | reliable recall: 0.1919
  confusion: [[19, 80], [25, 74]]

size 0, seed 2024 (zero-shot):
  macro-F1: 0.4254 | accuracy: 0.4697
  misinfo recall: 0.7475 | reliable recall: 0.1919
  confusion: [[19, 80], [25, 74]]


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss



size 20, seed 42:
  macro-F1: 0.6805 | accuracy: 0.6869
  misinfo recall: 0.8283 | reliable recall: 0.5455
  confusion: [[54, 45], [17, 82]]


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Step,Training Loss



size 100, seed 123:
  macro-F1: 0.8930 | accuracy: 0.8939
  misinfo recall: 0.7980 | reliable recall: 0.9899
  confusion: [[98, 1], [20, 79]]


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss



size 100, seed 2024:
  macro-F1: 0.8061 | accuracy: 0.8081
  misinfo recall: 0.7071 | reliable recall: 0.9091
  confusion: [[90, 9], [29, 70]]

=== SWEEP (mean over seeds) ===
English ceiling: 0.8647
MN   0 -> 0.4254 ± 0.0000
MN  20 -> 0.6719 ± 0.0203
MN  40 -> 0.5822 ± 0.2041
MN  80 -> 0.8623 ± 0.0236
MN 100 -> 0.8694 ± 0.0452


In [ ]:
import gc

constraint = english[english["source"].str.lower() == "constraint"]
pubhealth = english[english["source"].str.lower() == "pubhealth"]
pub_covid = pubhealth[pubhealth["text"].apply(has_covid)]
pub_other = pubhealth[~pubhealth["text"].apply(has_covid)]

en_covid = pd.concat([constraint, pub_covid])[["text", "label"]]
en_other_test = pub_other.reset_index(drop=True)
mn_covid = mongolian[mongolian["is_covid"] == 1][["text", "label"]]
mn_other_test = mongolian[mongolian["is_covid"] == 0].reset_index(drop=True)

train_df = pd.concat([en_covid, mn_covid], ignore_index=True)
print("COVID train:", len(train_df), "| EN-other test:", len(en_other_test), "| MN-other test:", len(mn_other_test))

en_scores, mn_scores = [], []
for seed in [42, 123, 2024]:
    m = train(train_df, seed)
    print("\nseed %d — English non-COVID:" % seed)
    en_scores.append(score(m, en_other_test))
    print("seed %d — Mongolian non-COVID:" % seed)
    mn_scores.append(score(m, mn_other_test))
    del m; gc.collect(); torch.cuda.empty_cache()

print("\n=== RQ4 (mean over seeds) ===")
print("English non-COVID: %.4f ± %.4f" % (np.mean(en_scores), np.std(en_scores)))
print("Mongolian non-COVID: %.4f ± %.4f" % (np.mean(mn_scores), np.std(mn_scores)))

COVID train: 965 | EN-other test: 357 | MN-other test: 284


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,0.641936
100,0.468983
150,0.358050
200,0.258674



seed 42 — English non-COVID:
  macro-F1: 0.5179 | accuracy: 0.5266
  misinfo recall: 0.9516 | reliable recall: 0.3004
  confusion: [[70, 163], [6, 118]]
seed 42 — Mongolian non-COVID:
  macro-F1: 0.8935 | accuracy: 0.8944
  misinfo recall: 0.9120 | reliable recall: 0.8805
  confusion: [[140, 19], [11, 114]]


...


seed 123 — English non-COVID:
  macro-F1: 0.5973 | accuracy: 0.5994
  misinfo recall: 0.9677 | reliable recall: 0.4034
  confusion: [[94, 139], [4, 120]]
seed 123 — Mongolian non-COVID:
  macro-F1: 0.9201 | accuracy: 0.9225
  misinfo recall: 0.8480 | reliable recall: 0.9811
  confusion: [[156, 3], [19, 106]]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,0.663115
100,0.540153
150,0.376089
200,0.280557



seed 2024 — English non-COVID:
  macro-F1: 0.5440 | accuracy: 0.5462
  misinfo recall: 0.8871 | reliable recall: 0.3648
  confusion: [[85, 148], [14, 110]]
seed 2024 — Mongolian non-COVID:
  macro-F1: 0.8802 | accuracy: 0.8838
  misinfo recall: 0.8080 | reliable recall: 0.9434
  confusion: [[150, 9], [24, 101]]

=== RQ4 (mean over seeds) ===
English non-COVID: 0.5531 ± 0.0330
Mongolian non-COVID: 0.8979 ± 0.0166
